# Fase 1+2 — Crawl L0→L2 (task 1.5) · **READY TO RUN DI LOKAL**

Notebook ini menjalankan crawl penuh DetikFinance dan menghasilkan **2 output** yang kamu minta,
tersimpan rapi dalam folder:

```
data/
├── raw/
│   ├── html_index/YYYY/MM/DD/page_n.html.gz   ← HTML halaman indeks (.gz)
│   └── html_article/{id[:3]}/{id}.html.gz     ← HTML artikel, 1 file = 1 artikel (.gz)
└── interim/
    ├── news_links.parquet / news_links.xlsx    ← TABEL LINK (output #1, buka di Excel)
    ├── news_clean_raw.parquet                   ← teks ter-parse dari HTML kandidat
    └── crawl_checkpoint.json                    ← progres (aman dihentikan & dilanjut)
```

Mapping link ↔ file HTML lewat `article_id` + kolom `html_path`.

## Sudah terverifikasi via pilot (23 hari Jan 2024)
- 2.048 link, 202 kandidat, 217 HTML `.gz`, **0 gagal parse**.
- Scraping isi berita dari `.gz` berhasil; mapping dua arah berfungsi.
- Checkpoint/resume berfungsi (hentikan kapan saja, jalankan lagi → lanjut).

---
## 0. Setup (jalankan sekali)

Dari root repo di terminal lokal:
```bash
python -m venv .venv && source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install -r requirements.txt
```
Lalu buka notebook ini dari **root repo** (bukan dari dalam folder `notebooks/`).

In [ ]:
# Pastikan working directory = ROOT repo (ada folder src/ dan config/)
import os, sys
from pathlib import Path

if Path("src").exists() and Path("config").exists():
    ROOT = Path(".").resolve()
elif Path("../src").exists():
    os.chdir("..")                 # jika dibuka dari dalam notebooks/
    ROOT = Path(".").resolve()
else:
    raise RuntimeError("Jalankan dari root repo (harus ada folder src/ dan config/).")
sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

In [ ]:
import yaml
from datetime import date
from src.crawl import Crawler, CrawlConfig
from src.pipeline_crawl import run_crawl, build_l1_rules, CrawlPaths

cfg = yaml.safe_load(open("config/config.yaml"))
aliases = yaml.safe_load(open("config/aliases.yaml"))
sel = cfg["selectors"]
print("Rentang di config:", cfg["date_range"])

## 1. Konfigurasi crawl

> ⚠️ **WAJIB:** ganti email kontak di `config/config.yaml` (`crawl.user_agent`) sebelum run.

> ⚠️ **Etika:** untuk crawl produksi penuh, gunakan `request_delay_sec = 1.0` (1 request/detik).
> Nilai lebih kecil hanya untuk uji cepat.

In [ ]:
l1 = build_l1_rules(aliases)
paths = CrawlPaths(raw=ROOT / "data/raw", interim=ROOT / "data/interim")

crawler = Crawler(cfg=CrawlConfig(
    user_agent=cfg["crawl"]["user_agent"],
    request_delay_sec=1.0,       # PRODUKSI = 1.0 (sopan). Uji cepat boleh 0.3-0.5
    jitter_sec=(0.5, 1.5),
))

## 2. Jalankan crawl

**Perkiraan waktu (Opsi A, 2024-01-02 → 2026-09-30, ~1.000 hari, 1 req/detik):**
indeks ~2–3 jam + artikel kandidat ~4–8 jam ≈ **6–11 jam total**.

Aman dibiarkan jalan di background terminal. Jika terputus, **jalankan lagi sel ini** —
checkpoint akan melanjutkan dari hari terakhir.

**Saran:** uji dulu 1 minggu (ubah `end`), pastikan output benar, baru jalankan rentang penuh.

In [ ]:
# --- UJI DULU: 1 minggu ---
start = date.fromisoformat(cfg["date_range"]["news_start"])   # 2024-01-02
end_uji = date(2024, 1, 8)

stats = run_crawl(crawler, sel, l1, paths, start, end_uji, seed=cfg["seed"])
print("\nUJI SELESAI:", stats)

In [ ]:
# --- RENTANG PENUH (bisa makan 6-11 jam; aman dicicil, lanjut via checkpoint) ---
# Sel ini MANDIRI: mendefinisikan start sendiri, jadi tidak perlu menjalankan sel UJI dulu.
start = date.fromisoformat(cfg["date_range"]["news_start"])    # 2024-01-02
end_penuh = date.fromisoformat(cfg["date_range"]["news_end"])   # 2026-09-30

stats = run_crawl(crawler, sel, l1, paths, start, end_penuh, seed=cfg["seed"])
print("PENUH SELESAI:", stats)

## 3. Verifikasi output

In [ ]:
import pandas as pd

links = pd.read_parquet(paths.links_parquet)
print("OUTPUT #1 — TABEL LINK")
print("  total link :", len(links))
print("  kandidat   :", int(links.is_candidate.sum()))
print("  html_saved :", int(links.html_saved.sum()))
print("  file Excel :", (paths.interim / 'news_links.xlsx').resolve())

n_html = len(list((paths.raw / 'html_article').rglob('*.html.gz')))
print("\nOUTPUT #2 — FILE HTML .gz:", n_html, "file artikel")
links.head()

In [ ]:
# Bukti scraping isi berita dari file .gz tersimpan (offline, tanpa akses server)
from src.crawl import load_html_gz
from src.parse import parse_article

row = links[links.html_saved].iloc[0]
art = parse_article(load_html_gz(paths.raw / row.html_path), row.url, sel)
print("LINK     :", row.url)
print("HTML .gz :", row.html_path)
print("judul    :", art.title)
print("published:", art.published_at, "| n_words:", art.n_words)
print("isi      :", art.body_text[:200], "...")